# Vietnamese ASR Training - Google Colab

**Nhan dang giong noi tieng Viet voi Wav2Vec2**

## Setup Checklist
- [ ] Runtime -> Change runtime type -> GPU (T4)
- [ ] Mount Google Drive
- [ ] Upload dataset len Drive
- [ ] Run all cells theo thu tu

## Latest Updates
- Fixed: Audio loading errors
- Fixed: processor NameError
- Fixed: Dataset columns mismatch
- Optimized: Symlink instead of copy (< 1s vs 5-15 min)

**Estimated Training Time: 15-20 hours on T4 GPU**

---
## PART 1: Environment Setup
---

## 1. Check GPU & Environment

Verify GPU availability and check environment information.

In [ ]:
import torch
import sys

print("=" * 60)
print("Environment Info")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\nGPU Ready!")
else:
    print("\nWARNING: GPU not available!")
    print("Go to: Runtime -> Change runtime type -> Hardware accelerator -> GPU")

## 2. Mount Google Drive

Mount Drive and create working directory structure.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create working directory
DRIVE_ROOT = "/content/drive/MyDrive/VietnameseASR"
os.makedirs(DRIVE_ROOT, exist_ok=True)

print(f"\nDrive mounted at: {DRIVE_ROOT}")
print("\nRecommended folder structure:")
print(f"{DRIVE_ROOT}/")
print("  |- data/               # Dataset files (train/val/test.jsonl)")
print("  |  |- train.jsonl")
print("  |  |- validation.jsonl")
print("  |  |- test.jsonl")
print("  |- vivos/              # Audio files folder")
print("  |  |- vivos/")
print("  |     |- train/waves/")
print("  |     |- test/waves/")
print("  |- models/             # Training checkpoints (auto-created)")
print("  |- final_model/        # Final trained model (auto-created)")

## 3. Install Dependencies

Install required Python packages for ASR training.

In [ ]:
%%capture
# Install packages silently (no output)
!pip install -q transformers datasets evaluate jiwer soundfile librosa accelerate tensorboard

In [ ]:
# Verify installation success
import transformers
import datasets
import evaluate

print("All packages installed successfully!")
print(f"  - transformers: {transformers.__version__}")
print(f"  - datasets: {datasets.__version__}")
print(f"  - evaluate: {evaluate.__version__}")

## 4. Clone Source Code from GitHub

Clone or update the Vietnamese ASR repository.

In [ ]:
import os
import sys

# Check if repository already exists
if os.path.exists('/content/Vietnamese-asr'):
    print("Repository already exists, updating...")
    os.chdir('/content/Vietnamese-asr')
    !git pull origin main
else:
    print("Cloning repository from GitHub...")
    !git clone https://github.com/CheeseThuong/Vietnamese-asr.git
    os.chdir('/content/Vietnamese-asr')

# CRITICAL: Add repository to Python path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"\nRepository ready!")
print(f"  Location: {os.getcwd()}")
print(f"  Python path: {sys.path[0]}")

# Verify repository structure
print(f"\nRepository structure:")
!ls -la
print(f"\nsrc/ contents:")
!ls -la src/

In [ ]:
# Verify Python imports (Diagnostic)
print("=" * 60)
print("Verifying imports...")
print("=" * 60)

try:
    from src.data.preprocessing import load_and_prepare_datasets
    from src.training.train_wav2vec2 import create_model, train_model
    print("All imports successful!")
except ModuleNotFoundError as e:
    print(f"Import failed: {e}")
    print("\nTroubleshooting:")
    print(f"  Current dir: {os.getcwd()}")
    print(f"  Python path: {sys.path[0]}")
    print(f"  src/ exists: {os.path.exists('src')}")

print("=" * 60)

---
## PART 2: Dataset Setup
---

## 5. Check Dataset Files

**IMPORTANT: Upload Instructions**

Before running this cell, you need to:

1. **On local machine**, run: `python convert_to_relative_paths.py`
   - This converts absolute paths to relative paths
   - Output files are in `processed_data_vivos/` folder

2. **Upload to Google Drive**:
   - Upload 3 files: `train.jsonl`, `validation.jsonl`, `test.jsonl`
   - Destination: `MyDrive/VietnameseASR/data/`

3. **Upload audio files**:
   - Upload entire `vivos/` folder
   - Destination: `MyDrive/VietnameseASR/vivos/`

**CRITICAL**: Files MUST have **relative paths** (e.g., `Data/vivos/vivos/train/...`)
NOT absolute paths (e.g., `D:\Projects\...`)

In [ ]:
from pathlib import Path
import json

# Dataset path on Google Drive
DATA_DIR = Path(f"{DRIVE_ROOT}/data")

# Check for required dataset files
required_files = ['train.jsonl', 'validation.jsonl', 'test.jsonl']
missing = [f for f in required_files if not (DATA_DIR / f).exists()]

if missing:
    print("ERROR: Missing dataset files:")
    for f in missing:
        print(f"  - {f}")
    print(f"\nExpected location: {DATA_DIR}")
    print("\nPlease upload dataset files to Google Drive first!")
    print("See instructions in the markdown cell above.")
else:
    print("SUCCESS: All dataset files found!")
    print()
    
    # Count samples and verify paths
    for file in required_files:
        filepath = DATA_DIR / file
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            count = len(lines)

        # Check sample path format
        if lines:
            sample = json.loads(lines[0])
            audio_path = sample['audio_path']
            is_relative = not os.path.isabs(audio_path)
            
            if is_relative:
                status = "OK"
                path_type = "relative"
            else:
                status = "ERROR"
                path_type = "ABSOLUTE"

            print(f"  [{status}] {file}: {count:,} samples ({path_type} paths)")
            print(f"         Sample: {audio_path[:60]}...")

            # Warning if absolute paths detected
            if not is_relative:
                print(f"         WARNING: Absolute paths detected!")
                print(f"         This will FAIL on Colab!")
                print(f"         Solution: Run 'python convert_to_relative_paths.py' locally")
                print()
        else:
            print(f"  [EMPTY] {file}: 0 samples")

## 6. Setup Audio Dataset

Create symlink to audio files from Google Drive (instant access, no copy needed).

In [ ]:
# Setup dataset in Colab workspace
print("Setting up audio dataset...")

# Create Data folder in Colab workspace
COLAB_DATA_DIR = Path("/content/Vietnamese-asr/Data")
COLAB_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Symlink vivos folder from Drive (FAST - instant access, no copy)
DRIVE_VIVOS = Path(f"{DRIVE_ROOT}/vivos")
COLAB_VIVOS = COLAB_DATA_DIR / "vivos"

# Verify audio folder exists on Drive
if not DRIVE_VIVOS.exists():
    print(f"\nERROR: Audio folder not found!")
    print(f"  Expected location: {DRIVE_VIVOS}")
    print(f"\nPlease upload 'vivos' folder to Google Drive:")
    print(f"  1. Download vivos dataset")
    print(f"  2. Upload to: {DRIVE_ROOT}/vivos/")
    print(f"  3. Expected structure: {DRIVE_ROOT}/vivos/vivos/train/waves/")
    raise FileNotFoundError(f"Audio dataset not found: {DRIVE_VIVOS}")

# Remove old copy/symlink if exists
if COLAB_VIVOS.exists() or COLAB_VIVOS.is_symlink():
    if COLAB_VIVOS.is_symlink():
        COLAB_VIVOS.unlink()
        print("Removed old symlink")
    else:
        import shutil
        shutil.rmtree(COLAB_VIVOS)
        print("Removed old copy")

# Create symlink (instant - no copying needed!)
print(f"\nCreating symlink to audio files...")
print(f"  Source: {DRIVE_VIVOS}")
print(f"  Destination: {COLAB_VIVOS}")

COLAB_VIVOS.symlink_to(DRIVE_VIVOS)
print(f"Symlink created successfully!")

# Verify audio files accessibility
if COLAB_VIVOS.exists():
    # Count audio files
    wav_files = list(COLAB_VIVOS.rglob("*.wav"))
    print(f"\nAudio files ready: {len(wav_files):,} WAV files")
    print(f"Location: {COLAB_VIVOS}")

    # Show sample structure
    print(f"\nSample file structure:")
    sample_files = list(COLAB_VIVOS.rglob("*.wav"))[:3]
    for f in sample_files:
        print(f"  {f.relative_to(COLAB_DATA_DIR)}")
else:
    print(f"\nERROR: Audio files verification failed!")
    raise FileNotFoundError(f"Cannot access: {COLAB_VIVOS}")

print("\nDataset setup complete!")

---
## PART 3: Training Configuration
---

## 7. Training Configuration

Configure training parameters optimized for Google Colab T4 GPU.

In [ ]:
import json

# Training configuration - Optimized for Colab T4 GPU
config = {
    # Model settings
    'pretrained_model': 'nguyenvulebinh/wav2vec2-base-vietnamese-250h',
    
    # Training parameters
    'num_train_epochs': 30,          # Total number of epochs
    'batch_size': 16,                # Batch size (T4 GPU ~ 16GB RAM)
    'gradient_accumulation_steps': 1, # Gradient accumulation
    'learning_rate': 3e-4,           # Learning rate
    
    # Optimization
    'use_fp16': True,                # Mixed precision training (faster)
    'apply_quantization': False,     # Don't quantize during training
    
    # Checkpointing
    'save_steps': 500,               # Save checkpoint every 500 steps
    'eval_steps': 500,               # Evaluate every 500 steps
}

# Output directories
OUTPUT_DIR = Path(f"{DRIVE_ROOT}/models/wav2vec2-vietnamese")
FINAL_MODEL_DIR = Path(f"{DRIVE_ROOT}/final_model")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save configuration for reference
with open(OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Training Configuration:")
print("=" * 60)
for key, value in config.items():
    print(f"  {key}: {value}")
print("=" * 60)
print(f"\nCheckpoints will be saved to: {OUTPUT_DIR}")
print(f"Final model will be saved to: {FINAL_MODEL_DIR}")
print(f"\nEstimated training time: 15-20 hours on T4 GPU")

---
## PART 4: Load Code & Data (CRITICAL SECTION)
---

## 8. Pull Latest Code & Reload Modules

**When to run this cell:**
- You updated code on GitHub and need latest bug fixes
- You got errors about missing columns or incorrect data format
- You're resuming training after a disconnect

**What this cell does:**
1. Pull latest code from GitHub
2. Clear old Python modules from memory
3. Reload modules with new code

**IMPORTANT:** After running this cell, you MUST re-run cell 9 (Load Datasets) to reload data with new code!

In [ ]:
# Pull latest code from GitHub and reload modules
import os
import sys

os.chdir('/content/Vietnamese-asr')

print("Pulling latest code from GitHub...")
!git pull origin main

print("\nReloading Python modules...")

# Remove old modules from Python's module cache
modules_to_reload = [
    'src.data.preprocessing',
    'src.training.train_wav2vec2'
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]
        print(f"  Cleared: {mod}")

# Re-import with new code
from src.data.preprocessing import load_and_prepare_datasets
from src.training.train_wav2vec2 import create_model, train_model

print("\nModules reloaded with new code!")
print("\n" + "=" * 60)
print("NEXT STEP: Run cell 9 to reload datasets with new code")
print("=" * 60)

## 9. Load Processor & Datasets

**CRITICAL CELL - MUST RUN:**
- First time setup
- After pulling new code (cell 8)
- After Colab disconnect/restart

**What this cell does:**
1. Load Wav2Vec2 processor (tokenizer + feature extractor)
2. Load and preprocess all datasets (train/val/test)
3. Convert audio to input_values and text to labels

**Time required:** 5-10 minutes for ~12,000 samples

**NOTE:** If you see errors about missing keys, re-run cell 8 first, then this cell.

In [ ]:
from transformers import Wav2Vec2Processor
from src.data.preprocessing import load_and_prepare_datasets

print("=" * 60)
print("Loading Processor & Datasets")
print("=" * 60)

# Load processor (tokenizer + feature extractor)
print("\n[1/2] Loading processor...")
processor = Wav2Vec2Processor.from_pretrained(config['pretrained_model'])
print(f"      Processor loaded: {config['pretrained_model']}")

# Change to repository directory (for relative paths)
os.chdir('/content/Vietnamese-asr')
print(f"\n[2/2] Loading datasets...")
print(f"      Working directory: {os.getcwd()}")
print(f"      This may take 5-10 minutes...")

# Load and preprocess datasets
train_dataset, val_dataset, test_dataset = load_and_prepare_datasets(
    str(DATA_DIR / 'train.jsonl'),
    str(DATA_DIR / 'validation.jsonl'),
    str(DATA_DIR / 'test.jsonl'),
    processor
)

print("\n" + "=" * 60)
print("Datasets loaded successfully!")
print("=" * 60)
print(f"  Train:      {len(train_dataset):>6,} samples")
print(f"  Validation: {len(val_dataset):>6,} samples")
print(f"  Test:       {len(test_dataset):>6,} samples")
print(f"  Total:      {len(train_dataset) + len(val_dataset) + len(test_dataset):>6,} samples")
print("=" * 60)

## 10. Verify Dataset Columns (DEBUG)

**Diagnostic cell to verify dataset structure before training.**

This cell checks:
- Dataset exists and is loaded
- Correct columns present (input_values, labels)
- No old columns present (audio_path, transcript, etc.)

**Expected output:**
```
Columns: ['input_values', 'labels']
OK: No old columns found
Dataset is ready for training!
```

**If you see old columns or MISSING errors:**
1. Re-run cell 8 (Pull code & Reload modules)
2. Re-run cell 9 (Load Processor & Datasets)
3. Re-run this cell to verify

In [ ]:
# DEBUG: Verify dataset columns before training
print("=" * 60)
print("Dataset Verification")
print("=" * 60)

# Check if dataset variable exists
try:
    train_dataset
except NameError:
    print("\nERROR: train_dataset is not defined!")
    print("\nSOLUTION:")
    print("  1. Scroll up to cell 9 (Load Processor & Datasets)")
    print("  2. Run that cell (takes 5-10 minutes)")
    print("  3. Come back and re-run this DEBUG cell")
    print("\n" + "=" * 60)
else:
    # Check train dataset structure
    print("\nTrain Dataset:")
    print(f"  Columns: {train_dataset.column_names}")
    print(f"  Features: {list(train_dataset.features.keys())}")
    print(f"  Number of samples: {len(train_dataset):,}")

    # Check sample structure
    if len(train_dataset) > 0:
        sample = train_dataset[0]
        print(f"\nSample Structure:")
        print(f"  Keys: {list(sample.keys())}")
        
        # Verify required columns
        if 'input_values' in sample:
            print(f"  [OK] input_values: shape {len(sample['input_values']):,}")
        else:
            print(f"  [ERROR] input_values: MISSING!")
            
        if 'labels' in sample:
            print(f"  [OK] labels: shape {len(sample['labels']):,}")
        else:
            print(f"  [ERROR] labels: MISSING!")

        # Verify NO old columns present
        old_columns = ['audio_path', 'transcript', 'speaker_id', 'dataset']
        found_old = [col for col in old_columns if col in sample]
        
        if found_old:
            print(f"\n[WARNING] Old columns still present: {found_old}")
            print("\nSOLUTION:")
            print("  1. Re-run cell 8 (Pull code & Reload modules)")
            print("  2. Re-run cell 9 (Load Processor & Datasets)")
            print("  3. Re-run this DEBUG cell to verify")
        else:
            print(f"\n[OK] No old columns found")
            print("\nDataset is ready for training!")

print("\n" + "=" * 60)

---
## PART 5: Model Training
---

## 11. Create Model

Initialize Wav2Vec2 model for CTC training.

In [ ]:
from src.training.train_wav2vec2 import create_model

print("=" * 60)
print("Creating Model")
print("=" * 60)

# Get vocabulary size from processor
vocab_size = len(processor.tokenizer)
print(f"\nVocabulary size: {vocab_size}")

# Create model
print(f"Loading pretrained model: {config['pretrained_model']}")
model = create_model(vocab_size, processor, config['pretrained_model'])

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model moved to: {device}")

# Model statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("\nModel Statistics:")
print("=" * 60)
print(f"  Total parameters:     {total_params:>15,}")
print(f"  Trainable parameters: {trainable_params:>15,}")
print(f"  Frozen parameters:    {frozen_params:>15,}")
print("=" * 60)
print("\nModel ready for training!")

In [ ]:
# Pre-training verification (MANDATORY CHECK)
print("=" * 70)
print("PRE-TRAINING VERIFICATION - COMPREHENSIVE CHECK")
print("=" * 70)

errors = []
warnings = []
info = []

# Check 1: Dataset exists and has valid size
print("\n[1/5] Checking dataset variables...")
dataset_ok = True
try:
    train_dataset
    val_dataset
    test_dataset
    print(f"      [OK] All datasets are loaded")
    print(f"           Train:      {len(train_dataset):>6,} samples")
    print(f"           Validation: {len(val_dataset):>6,} samples")
    print(f"           Test:       {len(test_dataset):>6,} samples")
    
    # Check if datasets are empty
    if len(train_dataset) == 0:
        errors.append("Training dataset is empty!")
        dataset_ok = False
    if len(val_dataset) == 0:
        warnings.append("Validation dataset is empty")
    
except NameError as e:
    errors.append(f"Dataset not loaded: {e}")
    print(f"      [ERROR] {e}")
    dataset_ok = False

# Check 2: Processor exists
print("\n[2/5] Checking processor...")
processor_ok = True
try:
    processor
    print("      [OK] Processor is loaded")
    
    # Check processor components
    if hasattr(processor, 'tokenizer'):
        vocab_size = len(processor.tokenizer)
        print(f"           Vocabulary size: {vocab_size}")
    if hasattr(processor, 'feature_extractor'):
        sampling_rate = processor.feature_extractor.sampling_rate
        print(f"           Sampling rate: {sampling_rate} Hz")
        
except NameError:
    errors.append("Processor not loaded")
    print("      [ERROR] Processor not loaded")
    processor_ok = False

# Check 3: Model exists
print("\n[3/5] Checking model...")
model_ok = True
try:
    model
    print("      [OK] Model is loaded")
    
    # Model device check
    model_device = next(model.parameters()).device
    print(f"           Device: {model_device}")
    
    # Model parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"           Total params: {total_params:,}")
    print(f"           Trainable: {trainable_params:,}")
    
except NameError:
    errors.append("Model not loaded")
    print("      [ERROR] Model not loaded")
    model_ok = False

# Check 4: Dataset structure (CRITICAL)
print("\n[4/5] Checking dataset structure...")
if dataset_ok and 'train_dataset' in dir():
    if len(train_dataset) > 0:
        sample = train_dataset[0]
        sample_keys = list(sample.keys())
        
        print(f"      Dataset columns: {sample_keys}")
        
        # Check for REQUIRED keys
        required_keys = ['input_values', 'labels']
        missing_keys = [key for key in required_keys if key not in sample]
        
        if missing_keys:
            errors.append(f"Dataset missing required keys: {missing_keys}")
            print(f"      [ERROR] Missing keys: {missing_keys}")
            print(f"              Current keys: {sample_keys}")
            print(f"              Expected keys: {required_keys}")
        else:
            print(f"      [OK] Dataset has correct structure")
            
            # Validate data types and shapes
            if 'input_values' in sample:
                input_shape = len(sample['input_values']) if hasattr(sample['input_values'], '__len__') else 'unknown'
                print(f"           input_values shape: {input_shape}")
                
            if 'labels' in sample:
                labels_shape = len(sample['labels']) if hasattr(sample['labels'], '__len__') else 'unknown'
                print(f"           labels shape: {labels_shape}")
        
        # Check for OLD columns (should NOT exist)
        old_columns = ['audio_path', 'transcript', 'speaker_id', 'dataset']
        found_old = [col for col in old_columns if col in sample]
        
        if found_old:
            errors.append(f"Dataset contains old columns: {found_old}")
            print(f"      [ERROR] Old columns found: {found_old}")
            print(f"              These columns should have been removed during preprocessing!")
        else:
            print(f"      [OK] No old columns found (dataset properly preprocessed)")
            
    else:
        errors.append("Training dataset is empty")
        print(f"      [ERROR] Training dataset has 0 samples")

# Check 5: Working directory and paths
print("\n[5/5] Checking working directory and paths...")
import os
current_dir = os.getcwd()
print(f"      Working directory: {current_dir}")

# Check if we're in the correct directory
if 'Vietnamese-asr' in current_dir or current_dir.endswith('Vietnamese-asr'):
    print(f"      [OK] Correct working directory")
else:
    warnings.append(f"Working directory may be incorrect: {current_dir}")
    print(f"      [WARNING] Expected to be in 'Vietnamese-asr' directory")

# Check Data folder exists
data_folder = os.path.join(current_dir, 'Data')
if os.path.exists(data_folder):
    print(f"      [OK] Data folder exists: {data_folder}")
else:
    warnings.append("Data folder not found in current directory")
    print(f"      [WARNING] Data folder not found: {data_folder}")

# Final verdict
print("\n" + "=" * 70)
if errors:
    print("VERIFICATION FAILED!")
    print("=" * 70)
    print(f"\n{len(errors)} ERROR(S) FOUND:")
    for i, error in enumerate(errors, 1):
        print(f"  {i}. {error}")
    
    print("\nSOLUTION STEPS:")
    print("  Step 1: Go to Cell 8 (Pull Latest Code & Reload Modules)")
    print("  Step 2: Run Cell 8 to pull latest code from GitHub")
    print("  Step 3: Go to Cell 9 (Load Processor & Datasets)")
    print("  Step 4: Run Cell 9 to reload datasets (takes 5-10 minutes)")
    print("  Step 5: Go to Cell 10 (Verify Dataset Columns)")
    print("  Step 6: Run Cell 10 to verify dataset structure")
    print("  Step 7: Go to Cell 11 (Create Model)")
    print("  Step 8: Run Cell 11 to recreate model")
    print("  Step 9: Come back to this cell and re-run verification")
    
    if warnings:
        print(f"\nALSO {len(warnings)} WARNING(S):")
        for i, warning in enumerate(warnings, 1):
            print(f"  {i}. {warning}")
    
    print("\n" + "=" * 70)
    print("DO NOT PROCEED TO TRAINING UNTIL ALL CHECKS PASS!")
    print("=" * 70)
    
    # Stop execution
    raise RuntimeError("Pre-training verification failed. Fix errors above before training.")
    
elif warnings:
    print("VERIFICATION PASSED (with warnings)")
    print("=" * 70)
    print(f"\n{len(warnings)} WARNING(S) FOUND:")
    for i, warning in enumerate(warnings, 1):
        print(f"  {i}. {warning}")
    print("\nThese warnings are not critical, but consider reviewing them.")
    print("You can proceed to training (next cell).")
    print("=" * 70)
else:
    print("VERIFICATION PASSED!")
    print("=" * 70)
    print("\nAll checks passed successfully!")
    print("\nREADY FOR TRAINING:")
    print("  - Datasets loaded and preprocessed correctly")
    print("  - Processor configured properly")
    print("  - Model initialized on GPU")
    print("  - Working directory correct")
    print(f"  - Total training samples: {len(train_dataset):,}")
    print(f"\nExpected training time: 15-20 hours on T4 GPU")
    print("\nYou can now proceed to Cell 12 (Start Training).")
    print("=" * 70)

## 11.5 Pre-Training Check (MANDATORY)

**CRITICAL: Run this cell before training!**

This cell performs final verification to prevent training errors.
If this check fails, DO NOT proceed to training - fix the issues first.

## 12. Start Training

**IMPORTANT INFORMATION:**

**Training Time:** 15-20 hours on T4 GPU

**Colab Limitations:**
- Free tier timeout: ~12 hours
- Pro tier timeout: ~24 hours

**Auto-save:** Checkpoints saved to Google Drive every 500 steps (safe if disconnected)

**What to do:**
1. Keep this browser tab OPEN
2. Don't close or refresh the page
3. Check progress every few hours
4. If timeout occurs, resume from latest checkpoint

**Monitoring:**
- Watch the progress bar
- Check loss values (should decrease)
- Evaluation runs every 500 steps

In [ ]:
from src.training.train_wav2vec2 import train_model

print("=" * 60)
print("Starting Training")
print("=" * 60)
print("\nIMPORTANT REMINDERS:")
print("  - Keep this browser tab OPEN")
print("  - Training time: 15-20 hours on T4 GPU")
print("  - Colab free timeout: ~12 hours")
print("  - Checkpoints auto-saved to Drive every 500 steps")
print("  - If timeout: Resume from latest checkpoint")
print("\n" + "=" * 60 + "\n")

# Start training
trainer = train_model(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    processor=processor,
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=config['num_train_epochs'],
    batch_size=config['batch_size'],
    gradient_accumulation_steps=config['gradient_accumulation_steps'],
    learning_rate=config['learning_rate'],
    use_fp16=config['use_fp16']
)

print("\n" + "=" * 60)
print("Training completed successfully!")
print("=" * 60)

## 13. Save Final Model

Save the trained model and processor to Google Drive.

In [ ]:
print("=" * 60)
print("Saving Final Model")
print("=" * 60)

# Save model
print(f"\nSaving model to: {FINAL_MODEL_DIR}")
trainer.save_model(str(FINAL_MODEL_DIR))
processor.save_pretrained(str(FINAL_MODEL_DIR))

# Save training history
import pandas as pd
if hasattr(trainer.state, 'log_history'):
    history_df = pd.DataFrame(trainer.state.log_history)
    history_csv = f"{DRIVE_ROOT}/training_history.csv"
    history_df.to_csv(history_csv, index=False)
    print(f"Training history saved to: {history_csv}")

print("\n" + "=" * 60)
print("Training Completed!")
print("=" * 60)
print(f"\nFinal model saved to: {FINAL_MODEL_DIR}")
print("\nYou can now:")
print("  1. Download model from Google Drive to local machine")
print("  2. Use model directly from Drive in other notebooks")
print("  3. Upload model to HuggingFace Hub for public access")
print("  4. Test model with audio files (see cells below)")
print("\n" + "=" * 60)

---
## PART 6: Monitoring & Testing (Optional)
---

## 14. Monitor Training Progress (Optional)

Run these cells DURING training to monitor progress.

In [ ]:
# TensorBoard - visualize training metrics
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/runs

In [ ]:
# GPU monitoring - check GPU usage and memory
!nvidia-smi

In [ ]:
# Check latest checkpoints
!ls -lht {OUTPUT_DIR}/checkpoint-*/ | head -10

## 15. Test Trained Model (After Training)

Load and test the trained model with audio files.

In [ ]:
# Load trained model for inference
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import soundfile as sf

print("Loading trained model...")
model = Wav2Vec2ForCTC.from_pretrained(str(FINAL_MODEL_DIR))
processor = Wav2Vec2Processor.from_pretrained(str(FINAL_MODEL_DIR))
model = model.to(device)
model.eval()

print("Model loaded successfully!")
print(f"Device: {device}")

In [ ]:
# Transcribe function
def transcribe(audio_path):
    """
    Transcribe audio file to text using trained model.
    
    Args:
        audio_path: Path to audio file (.wav format, 16kHz recommended)
    
    Returns:
        transcription: Transcribed text
    """
    # Load audio file
    speech, sample_rate = sf.read(audio_path)
    
    # Process audio
    inputs = processor(
        speech, 
        sampling_rate=16000, 
        return_tensors="pt", 
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate predictions
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # Decode predictions
    pred_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(pred_ids)[0]
    
    return transcription

# Example usage (uncomment and modify path):
# audio_file = "/content/drive/MyDrive/VietnameseASR/test_audio.wav"
# result = transcribe(audio_file)
# print(f"Transcription: {result}")

---
## PART 7: Tips & Troubleshooting
---

## 16. Tips & Best Practices

### Handling Colab Timeout

**Problem:** Training takes 15-20 hours, but Colab free tier disconnects after ~12 hours

**Solution 1: Split Training into Multiple Sessions**
```python
# Session 1: Train 10 epochs
config['num_train_epochs'] = 10
# Run training...

# Session 2: Resume from checkpoint and train 10 more epochs
config['resume_from_checkpoint'] = str(OUTPUT_DIR / 'checkpoint-5000')
config['num_train_epochs'] = 20  # Total epochs
# Run training again...
```

**Solution 2: Upgrade to Colab Pro**
- Timeout: ~24 hours (enough for full training)
- Better GPU: A100/V100 (faster training ~8-10 hours)
- Higher RAM: 50GB+ (can use larger batch sizes)

### Auto-save Feature

Checkpoints are automatically saved to Google Drive every 500 steps.
If Colab disconnects:
1. Reconnect and re-run setup cells (1-9)
2. Resume training from latest checkpoint
3. Model training will continue from where it stopped

### Optimizing Training Speed

1. **Use FP16 (Mixed Precision):** Already enabled in config (use_fp16=True)
2. **Increase Batch Size:** If you have Pro tier with more GPU RAM
3. **Reduce Evaluation Frequency:** Change eval_steps from 500 to 1000
4. **Use Gradient Accumulation:** Simulate larger batch sizes

### Common Issues

**Issue 1: GPU Not Available**
- Solution: Runtime -> Change runtime type -> Hardware accelerator -> GPU

**Issue 2: Drive Space Full**
- Solution: Clean up old checkpoints, keep only last 2-3
- Command: `!rm -rf {OUTPUT_DIR}/checkpoint-1000`

**Issue 3: Out of Memory (OOM)**
- Solution: Reduce batch_size from 16 to 8 or 4
- Increase gradient_accumulation_steps to maintain effective batch size

**Issue 4: Training Loss Not Decreasing**
- Check: Learning rate (try 1e-4 or 5e-4)
- Check: Data quality (verify audio files load correctly)
- Check: Model weights (ensure fine-tuning from pretrained)

### Monitoring Training

Best practices:
- Check TensorBoard every few hours
- Monitor loss values (should decrease steadily)
- Check evaluation metrics (WER should decrease)
- Verify checkpoints are saving to Drive

### After Training

1. **Evaluate on test set:** Use test_dataset to measure final performance
2. **Compare with baseline:** Check if model beats pretrained model
3. **Save best checkpoint:** Not just the final one
4. **Document results:** Save metrics and configuration for reproducibility

---

## Questions or Issues?

If you encounter problems:
1. Check the DEBUG cell outputs
2. Verify all files uploaded correctly to Drive
3. Ensure relative paths (not absolute) in JSONL files
4. Check GitHub repository for latest updates
5. Review error messages carefully

---

**End of Notebook**